# Hybrid Pawar + V3 — Devyansh Routes (Route_1 – Route_6)

### What this notebook does
1. **Pawar ANN** (trained on India data) flags candidate windows from Devyansh's Route_1–Route_6
2. **V3-style multi-sensor features** — Z-axis dip/bump, gyroscope, accel-X, speed context
3. **Z-axis kurtosis** — key discriminator (sharp impulsive = pothole, gradual = bump)
4. **Improved classification** — balanced pothole/bump detection using kurtosis + gyro + physics
5. **Leaflet map** with color-coded markers, Google Maps links, route polylines

### Physics of pothole vs bump
- **Pothole**: wheel drops → sharp V-shaped Z dip → bounce-back → **high kurtosis** (impulsive)
- **Road bump**: wheel climbs gradually → gentle Z rise → settle back → **low kurtosis** (smooth)
- Both create dip AND bump in Z, but the **sharpness** (kurtosis) and **vehicle rocking** (gyro) differ

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE
from scipy.stats import kurtosis as sp_kurtosis
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import folium

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
print('Libraries loaded.')

Libraries loaded.


## 1. Train Pawar ANN on India Dataset

In [2]:
# Load India windowed features (from pawar/ folder one level up)
windowed = pd.read_csv(os.path.join('..', 'pawar', 'pawar_windowed_features.csv'))
feature_cols = [c for c in windowed.columns if c.endswith(('_min', '_max', '_mean', '_std'))]
print(f'India data: {windowed.shape}  |  {len(feature_cols)} features')
print(f'Pothole: {windowed["is_pothole"].sum()} / {len(windowed)} '
      f'({windowed["is_pothole"].mean()*100:.1f}%)')

India data: (217, 20)  |  16 features
Pothole: 98 / 217 (45.2%)


In [3]:
# Pawar ANN architecture
class PawarANN(nn.Module):
    def __init__(self, n_features, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(64, 32),  nn.ReLU(), nn.Dropout(dropout_rate),
            nn.Linear(32, 1),   nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# Split + SMOTE + Train
X = windowed[feature_cols].values
y = windowed['is_pothole'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train_sc, y_train)
print(f'After SMOTE: {len(y_sm)} ({y_sm.sum()} pothole)')

EPOCHS, BATCH = 140, 3
model = PawarANN(X_train_sc.shape[1]).to(device)
train_ds = TensorDataset(torch.FloatTensor(X_sm).to(device),
                          torch.FloatTensor(y_sm).to(device))
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

model.eval()
with torch.no_grad():
    tp = (model(torch.FloatTensor(X_test_sc).to(device)) >= 0.5).cpu().numpy()
print(f'Test acc: {accuracy_score(y_test, tp):.4f}  F1: {f1_score(y_test, tp):.4f}')
print('Pawar model trained.')

After SMOTE: 190 (95 pothole)
Test acc: 0.9545  F1: 0.9474
Pawar model trained.


## 2. Load Devyansh Route_1 – Route_6 Data

In [14]:
# Devyansh routes are siblings of this notebook file
folders  = ['Route_1', 'Route_2', 'Route_3', 'Route_4', 'Route_5', 'Route_6']
DATA_ROOT = '.'           # notebook lives inside Devyansh/

our_data = {}
for folder in folders:
    base = os.path.join(DATA_ROOT, folder)
    our_data[folder] = {
        'accel':    pd.read_csv(os.path.join(base, 'Accelerometer.csv')),
        'total':    pd.read_csv(os.path.join(base, 'TotalAcceleration.csv')),
        'location': pd.read_csv(os.path.join(base, 'Location.csv')),
        'gyro':     pd.read_csv(os.path.join(base, 'Gyroscope.csv')),
    }
    d = our_data[folder]
    print(f'{folder}:  accel={len(d["accel"]):,}  total={len(d["total"]):,}  '
          f'gyro={len(d["gyro"]):,}  loc={len(d["location"]):,}')

Route_1:  accel=15,158  total=15,167  gyro=15,173  loc=152
Route_2:  accel=31,981  total=32,001  gyro=32,040  loc=325
Route_3:  accel=5,269  total=5,278  gyro=5,283  loc=51
Route_4:  accel=23,565  total=23,572  gyro=23,576  loc=239
Route_5:  accel=10,390  total=10,394  gyro=10,392  loc=105
Route_6:  accel=17,812  total=17,812  gyro=17,823  loc=176


## 3. Create 2-Second Windowed Features + Pawar Inference

In [15]:
WINDOW_SEC = 2

def create_windows(total_df, loc_df, route_name, window_sec=WINDOW_SEC):
    """Create Pawar-style 2-second windows from TotalAcceleration."""
    t = total_df['seconds_elapsed'].values
    loc_t   = loc_df['seconds_elapsed'].values.astype('float64')
    loc_lat = loc_df['latitude'].values.astype('float64')
    loc_lon = loc_df['longitude'].values.astype('float64')
    loc_spd = (loc_df['speed'].values.astype('float64')
               if 'speed' in loc_df.columns else np.zeros(len(loc_df)))

    windows, t_start = [], t[0]
    while t_start + window_sec <= t[-1] + 1:
        mask = (t >= t_start) & (t < t_start + window_sec)
        w = total_df[mask]
        if len(w) < 3:
            t_start += window_sec; continue
        row = {}
        for axis in ['x', 'y', 'z']:
            v = w[axis].values.astype(float)
            ax = axis.upper()
            row[f'{ax}_min']  = v.min();  row[f'{ax}_max']  = v.max()
            row[f'{ax}_mean'] = v.mean(); row[f'{ax}_std']  = v.std() if len(v) > 1 else 0.0
        mag = np.sqrt(w['x'].values**2 + w['y'].values**2 + w['z'].values**2)
        row['mag_min']  = mag.min();  row['mag_max']  = mag.max()
        row['mag_mean'] = mag.mean(); row['mag_std']  = mag.std() if len(mag) > 1 else 0.0
        mid = t_start + window_sec / 2
        row['latitude']     = float(np.interp(mid, loc_t, loc_lat))
        row['longitude']    = float(np.interp(mid, loc_t, loc_lon))
        row['speed']        = float(np.interp(mid, loc_t, loc_spd))
        row['n_samples']    = len(w)
        row['route']        = route_name
        row['window_start'] = t_start
        windows.append(row)
        t_start += window_sec
    return pd.DataFrame(windows)


our_all_windowed = pd.concat(
    [create_windows(our_data[f]['total'], our_data[f]['location'], f) for f in folders],
    ignore_index=True)
our_feature_cols = [c for c in our_all_windowed.columns
                    if c.endswith(('_min','_max','_mean','_std'))]
print(f'Total windows: {len(our_all_windowed)}')
for f in folders:
    print(f'  {f}: {(our_all_windowed["route"]==f).sum()}')

Total windows: 521
  Route_1: 76
  Route_2: 160
  Route_3: 26
  Route_4: 118
  Route_5: 52
  Route_6: 89


In [16]:
# Pawar inference on Devyansh windows
our_scaler = StandardScaler()
our_X_scaled = our_scaler.fit_transform(
    our_all_windowed[our_feature_cols].values)

model.eval()
with torch.no_grad():
    our_probs = model(
        torch.FloatTensor(our_X_scaled).to(device)).cpu().numpy()

our_all_windowed = our_all_windowed.copy()
our_all_windowed['pawar_prob'] = our_probs
our_all_windowed['pawar_pred'] = (our_probs >= 0.5).astype(int)

print(f'Pawar flagged: {our_all_windowed["pawar_pred"].sum()} / '
      f'{len(our_all_windowed)} '
      f'({our_all_windowed["pawar_pred"].mean()*100:.1f}%)')
for f in folders:
    m = our_all_windowed['route'] == f
    n = our_all_windowed.loc[m, 'pawar_pred'].sum()
    print(f'  {f}: {n}/{m.sum()} ({n/m.sum()*100:.1f}%)')

Active windows (Z_std >= 0.5): 472 / 521 (90.6%)
  Route_1: 63/76 (82.9%)
  Route_2: 156/160 (97.5%)
  Route_3: 7/26 (26.9%)
  Route_4: 113/118 (95.8%)
  Route_5: 52/52 (100.0%)
  Route_6: 81/89 (91.0%)


In [17]:
# Speed context: detect vehicle start/stop phases from GPS
def compute_speed_context(gyro_df, loc_df, lookback_sec=3.0):
    sensor_t  = gyro_df['seconds_elapsed'].values.astype('float64')
    gps_t     = loc_df['seconds_elapsed'].values.astype('float64')
    gps_speed = loc_df['speed'].values.astype('float64')
    interp_speed = np.interp(sensor_t, gps_t, gps_speed)
    is_start = np.zeros(len(sensor_t), dtype=bool)
    is_stop  = np.zeros(len(sensor_t), dtype=bool)
    for i in range(len(sensor_t)):
        t_now = sensor_t[i]
        mb = (sensor_t >= t_now - lookback_sec) & (sensor_t <= t_now)
        if interp_speed[mb].min() < 1.0: is_start[i] = True
        mf = (sensor_t >= t_now) & (sensor_t <= t_now + lookback_sec)
        if interp_speed[mf].min() < 1.0: is_stop[i] = True
    return pd.DataFrame({'seconds_elapsed': sensor_t,
                         'speed': interp_speed,
                         'is_start_phase': is_start,
                         'is_stop_phase':  is_stop})

speed_contexts = {}
for f in folders:
    ctx = compute_speed_context(our_data[f]['gyro'], our_data[f]['location'])
    speed_contexts[f] = ctx
    print(f'{f}: speed {ctx["speed"].min()*3.6:.1f} - '
          f'{ctx["speed"].max()*3.6:.1f} km/h')

Route_1: speed 0.0 - 38.7 km/h
Route_2: speed 0.0 - 76.0 km/h
Route_3: speed 0.0 - 8.1 km/h
Route_4: speed 0.0 - 45.7 km/h
Route_5: speed 0.1 - 38.0 km/h
Route_6: speed 0.0 - 35.5 km/h


## 4. V3 Feature Extraction (with Z-axis Kurtosis)

**Z-axis kurtosis** measures the sharpness of the vertical-axis signal in each window.
- **High kurtosis (> 3)**: Sharp, impulsive event → pothole (sudden drop + bounce-back)
- **Low kurtosis (< 2)**: Gradual, smooth oscillation → speed bump / road bump

In [18]:
MIN_SPEED = 2.0  # m/s — ignore near-stationary events

def extract_v3_features(route_win, route_data, speed_ctx,
                        pawar_threshold=0.4):
    """
    Extract multi-sensor physics features for each Pawar-flagged window.
    z_kurtosis: excess kurtosis of Z values — high = sharp (pothole), low = gradual (bump).
    """
    gyro_df  = route_data['gyro']
    accel_df = route_data['accel']
    total_df = route_data['total']
    loc_df   = route_data['location']

    loc_t   = loc_df['seconds_elapsed'].values.astype('float64')
    loc_lat = loc_df['latitude'].values.astype('float64')
    loc_lon = loc_df['longitude'].values.astype('float64')

    events = []
    rejects = {'low_speed': 0, 'start': 0, 'stop': 0, 'below_thr': 0}

    for _, win in route_win.iterrows():
        p = win['pawar_prob']
        if p < pawar_threshold:
            rejects['below_thr'] += 1; continue

        t0 = win['window_start']
        t1 = t0 + WINDOW_SEC
        mid = (t0 + t1) / 2

        seg_ctx = speed_ctx[
            (speed_ctx['seconds_elapsed'] >= t0) &
            (speed_ctx['seconds_elapsed'] <= t1)]
        if len(seg_ctx) == 0:
            idx = (speed_ctx['seconds_elapsed'] - mid).abs().idxmin()
            seg_ctx = speed_ctx.iloc[[idx]]

        avg_speed = float(seg_ctx['speed'].mean())
        if avg_speed < MIN_SPEED:           rejects['low_speed'] += 1; continue
        if seg_ctx['is_start_phase'].any(): rejects['start']     += 1; continue
        if seg_ctx['is_stop_phase'].any():  rejects['stop']      += 1; continue

        # TotalAccel Z features
        tm = ((total_df['seconds_elapsed'] >= t0) &
              (total_df['seconds_elapsed'] < t1))
        t_seg = total_df[tm]
        if len(t_seg) < 5:
            continue

        z = t_seg['z'].values.astype(float)
        gravity = 9.81

        dip_mag  = max(0.0, gravity - z.min())
        bump_mag = max(0.0, z.max() - gravity)
        total_z_dev = float(np.abs(z - gravity).max())
        z_std = float(z.std()) if len(z) > 1 else 0.0
        z_kurt = float(sp_kurtosis(z, fisher=True))   # excess kurtosis

        # Gyroscope features
        gm = ((gyro_df['seconds_elapsed'] >= t0) &
              (gyro_df['seconds_elapsed'] < t1))
        g_seg = gyro_df[gm]
        if len(g_seg) > 0:
            gmag = np.sqrt(g_seg['x']**2 + g_seg['y']**2 + g_seg['z']**2)
            gyro_peak = float(gmag.max())
            gyro_rms  = float(gmag.mean())
        else:
            gyro_peak = gyro_rms = 0.0

        # Accelerometer X features
        am = ((accel_df['seconds_elapsed'] >= t0) &
              (accel_df['seconds_elapsed'] < t1))
        a_seg = accel_df[am]
        if len(a_seg) > 0:
            accel_x_min  = float(a_seg['x'].min())
            accel_x_max  = float(a_seg['x'].max())
            accel_x_peak = float(a_seg['x'].abs().max())
        else:
            accel_x_min = accel_x_max = accel_x_peak = 0.0

        # Braking context
        pre  = accel_df[(accel_df['seconds_elapsed'] >= t0 - 1.0) &
                        (accel_df['seconds_elapsed'] < t0)]
        post = accel_df[(accel_df['seconds_elapsed'] > t1) &
                        (accel_df['seconds_elapsed'] <= t1 + 1.0)]
        braking_before = bool(len(pre)  > 0 and float(pre['x'].min())  < -1.5)
        braking_after  = bool(len(post) > 0 and float(post['x'].min()) < -1.5)

        events.append({
            'window_start':   t0,
            'duration':       WINDOW_SEC,
            'latitude':       float(np.interp(mid, loc_t, loc_lat)),
            'longitude':      float(np.interp(mid, loc_t, loc_lon)),
            'speed':          avg_speed,
            'pawar_prob':     p,
            'z_min': float(z.min()), 'z_max': float(z.max()),
            'z_mean': float(z.mean()), 'z_std': z_std,
            'dip_magnitude':  dip_mag,
            'bump_magnitude': bump_mag,
            'total_z_dev':    total_z_dev,
            'z_kurtosis':     z_kurt,
            'gyro_peak':      gyro_peak,
            'gyro_rms':       gyro_rms,
            'accel_x_min':    accel_x_min,
            'accel_x_max':    accel_x_max,
            'accel_x_peak':   accel_x_peak,
            'braking_before': braking_before,
            'braking_after':  braking_after,
            'est_size_m':     avg_speed * WINDOW_SEC,
        })

    return pd.DataFrame(events), rejects


# Process all Devyansh routes
all_events = {}
for f in folders:
    route_win = our_all_windowed[our_all_windowed['route'] == f]
    df, rej = extract_v3_features(
        route_win, our_data[f], speed_contexts[f], pawar_threshold=0.4)
    all_events[f] = df
    n_flag = (route_win['pawar_prob'] >= 0.4).sum()
    print(f'{f}: {n_flag} flagged -> {len(df)} after filtering  '
          f'(low_spd={rej["low_speed"]} start={rej["start"]} stop={rej["stop"]})')

total_cand = sum(len(d) for d in all_events.values())
print(f'\nTotal candidates: {total_cand}')

Route_1: 63 active -> 38 after filtering  (low_act=13 low_spd=15 start=5 stop=5)
Route_2: 156 active -> 121 after filtering  (low_act=4 low_spd=21 start=7 stop=7)
Route_3: 7 active -> 0 after filtering  (low_act=19 low_spd=6 start=1 stop=0)
Route_4: 113 active -> 83 after filtering  (low_act=5 low_spd=16 start=8 stop=6)
Route_5: 52 active -> 43 after filtering  (low_act=0 low_spd=1 start=5 stop=3)
Route_6: 81 active -> 73 after filtering  (low_act=8 low_spd=1 start=5 stop=2)

Total candidates: 358


## 5. Improved Hybrid Classification

**Classification priority order:**
1. Hard Braking / Acceleration — X-accel dominant, low gyro, low Z deviation
2. Pothole (high confidence) — `gyro > 0.5` OR `(gyro > 0.3 AND kurtosis > 5)`
3. Road Bump — bump-only (Z never fell below gravity significantly)
4. Speed Bump — dip + bump, but `gyro < 0.3 AND kurtosis < 2` (gentle & gradual)
5. Pothole (moderate) — `gyro > 0.3 AND kurtosis > 3` OR `kurtosis > 5 AND dip > 2`
6. Road Bump / Speed Bump — gradual oscillation, low kurtosis
7. Rough Patch — mild dip, low energy
8. Possible Pothole — ML-only (high Pawar prob, weak physics)

In [19]:
def classify_hybrid(row):
    """
    Improved Pawar+V3 hybrid classifier using kurtosis + gyro + physics.
    Z-axis kurtosis separates sharp impulsive events (potholes, kurtosis >> 3)
    from gradual oscillations (bumps, kurtosis < 2).
    """
    p       = row['pawar_prob']
    dip     = row['dip_magnitude']
    bump    = row['bump_magnitude']
    tz_dev  = row['total_z_dev']
    z_kurt  = row['z_kurtosis']
    g_peak  = row['gyro_peak']
    x_peak  = row['accel_x_peak']
    x_min   = row['accel_x_min']
    x_max   = row['accel_x_max']
    brk_bef = row['braking_before']
    brk_aft = row['braking_after']

    has_dip  = dip  > 0.5
    has_bump = bump > 0.5

    # 1. HARD BRAKING / ACCELERATION
    if x_peak > 2.0 and g_peak < 0.3 and tz_dev < 2.0:
        sev = min(10, x_peak * 1.5)
        if x_min < -2.5:
            return 'Hard Braking', sev, 0.80
        if x_max >  2.5:
            return 'Acceleration', sev, 0.75
        lbl = 'Hard Braking' if abs(x_min) > x_max else 'Acceleration'
        return lbl, sev, 0.70

    if brk_bef and g_peak < 0.3 and tz_dev < 2.0:
        return 'Hard Braking', min(10, x_peak * 1.2), 0.65

    # 2. POTHOLE — high confidence
    if has_dip and (g_peak > 0.5 or (g_peak > 0.3 and z_kurt > 5)):
        sev = min(10, dip * 1.0 + g_peak * 2.0 + min(z_kurt, 10) * 0.2)
        conf = min(0.95, 0.55 + dip * 0.03 + g_peak * 0.15
                   + p * 0.1 + min(z_kurt, 10) * 0.01)
        if brk_aft:
            conf = min(1.0, conf + 0.05)
        return 'Pothole', sev, conf

    if has_dip and dip > 5.0 and z_kurt > 3:
        sev = min(10, dip * 0.8 + min(z_kurt, 10) * 0.3)
        return 'Pothole', sev, min(0.80, 0.50 + dip * 0.02 + p * 0.1)

    # 3. ROAD BUMP — bump-only
    if has_bump and not has_dip:
        sev = min(10, bump * 1.5)
        return ('Road Bump', sev, 0.70) if bump > 2.0 else ('Road Bump', sev, 0.55)

    # 4. SPEED BUMP — both dip + bump, gentle & gradual
    if has_dip and has_bump and g_peak < 0.3 and z_kurt < 2:
        total_osc = dip + bump
        sev = min(10, total_osc * 0.7)
        return ('Speed Bump', sev, 0.65) if total_osc > 3.0 else ('Road Bump', sev, 0.50)

    # 5. POTHOLE — moderate confidence
    if has_dip and g_peak > 0.3 and z_kurt > 3:
        sev = min(10, dip * 0.8 + g_peak * 2.0 + p * 1.0)
        return 'Pothole', sev, min(0.80, 0.45 + g_peak * 0.15 + p * 0.1)

    if has_dip and z_kurt > 5 and dip > 2.0:
        sev = min(10, dip * 0.8 + min(z_kurt, 10) * 0.3)
        return 'Pothole', sev, min(0.70, 0.40 + p * 0.15 + dip * 0.02)

    if has_dip and p > 0.8 and dip > 1.5 and z_kurt > 3:
        sev = min(10, dip * 0.8 + p * 2.0)
        return 'Pothole', sev, min(0.70, 0.40 + p * 0.15)

    # 6. ROAD BUMP / SPEED BUMP — gradual oscillation
    if has_dip and has_bump and z_kurt < 3 and g_peak < 0.5:
        total_osc = dip + bump
        sev = min(10, total_osc * 0.6)
        if total_osc > 4.0:
            return 'Speed Bump', sev, 0.55
        if bump >= dip * 0.7:
            return 'Road Bump', sev, 0.45
        return 'Rough Patch', min(5, total_osc * 0.4), 0.35

    # 7. SPEED BUMP catchall
    if has_dip and has_bump and g_peak > 0.3 and z_kurt < 3:
        return 'Speed Bump', min(10, (dip + bump) * 0.6), 0.50

    # 8. ROUGH PATCH
    if has_dip:
        return 'Rough Patch', min(5, dip * 0.8), 0.30
    if has_bump:
        return 'Road Bump', min(5, bump * 0.8), 0.35

    # 9. POSSIBLE POTHOLE — ML confident, physics ambiguous
    if p > 0.8:
        return 'Possible Pothole', min(6, tz_dev * 0.8 + p * 2), 0.40

    # 10. MINOR ANOMALY
    return 'Minor Anomaly', min(4, tz_dev * 0.5), 0.20


# Apply classification to all routes
for f in folders:
    df = all_events[f]
    if len(df) == 0: continue
    cls = df.apply(classify_hybrid, axis=1, result_type='expand')
    df['event_type']  = cls[0]
    df['severity']    = cls[1].astype('float32')
    df['confidence']  = cls[2].astype('float32')
    df['route']       = f
    all_events[f]     = df

all_classified = pd.concat(
    [all_events[f] for f in folders if len(all_events[f]) > 0],
    ignore_index=True)

print('=== CLASSIFICATION RESULTS ===')
print(all_classified['event_type'].value_counts().to_string())

print('\n=== PER ROUTE ===')
for f in folders:
    df = all_events[f]
    if len(df) == 0: continue
    print(f'\n{f} ({len(df)} events):')
    for et in df['event_type'].value_counts().index:
        sub = df[df['event_type'] == et]
        print(f'  {et:20s}: {len(sub):3d}  '
              f'avg_sev={sub["severity"].mean():.1f}  '
              f'avg_gyro={sub["gyro_peak"].mean():.3f}  '
              f'avg_kurt={sub["z_kurtosis"].mean():.1f}')

print('\n=== KURTOSIS VALIDATION ===')
for et in all_classified['event_type'].value_counts().index:
    sub = all_classified[all_classified['event_type'] == et]
    print(f'{et:20s}  n={len(sub):3d}  '
          f'avg_kurt={sub["z_kurtosis"].mean():.2f}  '
          f'avg_gyro={sub["gyro_peak"].mean():.3f}  '
          f'avg_dip={sub["dip_magnitude"].mean():.2f}  '
          f'avg_bump={sub["bump_magnitude"].mean():.2f}')

=== CLASSIFICATION RESULTS ===
event_type
Speed Bump      205
Pothole         147
Road Bump         4
Hard Braking      2

=== PER ROUTE ===

Route_1 (38 events):
  Speed Bump          :  31  avg_sev=5.8  avg_gyro=0.262  avg_kurt=0.1
  Pothole             :   7  avg_sev=9.2  avg_gyro=1.401  avg_kurt=1.0

Route_2 (121 events):
  Speed Bump          :  69  avg_sev=6.2  avg_gyro=0.298  avg_kurt=0.3
  Pothole             :  49  avg_sev=8.2  avg_gyro=1.590  avg_kurt=3.3
  Road Bump           :   3  avg_sev=1.9  avg_gyro=0.364  avg_kurt=0.2

Route_4 (83 events):
  Speed Bump          :  59  avg_sev=4.7  avg_gyro=0.286  avg_kurt=0.3
  Pothole             :  21  avg_sev=8.0  avg_gyro=0.745  avg_kurt=1.1
  Hard Braking        :   2  avg_sev=2.0  avg_gyro=0.203  avg_kurt=0.1
  Road Bump           :   1  avg_sev=2.0  avg_gyro=0.319  avg_kurt=0.3

Route_5 (43 events):
  Speed Bump          :  22  avg_sev=5.2  avg_gyro=0.335  avg_kurt=0.3
  Pothole             :  21  avg_sev=8.9  avg_gyro=1.667  av

## 6. DBSCAN Clustering + Map

In [20]:
# DBSCAN: cluster nearby events of the same type within 20 m
interesting = all_classified[
    all_classified['event_type'] != 'Minor Anomaly'].copy()
print(f'Events for map: {len(interesting)}  '
      f'(dropped {len(all_classified)-len(interesting)} Minor Anomalies)')

cluster_id = 0
interesting['cluster'] = -1
for etype in interesting['event_type'].unique():
    mask = interesting['event_type'] == etype
    sub  = interesting[mask]
    if len(sub) < 2:
        interesting.loc[mask, 'cluster'] = cluster_id
        cluster_id += 1; continue
    coords_rad = np.radians(sub[['latitude','longitude']].values)
    db = DBSCAN(eps=20/6371000, min_samples=1,
                metric='haversine').fit(coords_rad)
    interesting.loc[mask, 'cluster'] = db.labels_ + cluster_id
    cluster_id += db.labels_.max() + 2

print(f'Clusters: {interesting["cluster"].nunique()}')
print(f'\nFinal distribution:')
print(interesting['event_type'].value_counts().to_string())

Events for map: 358  (dropped 0 Minor Anomalies)
Clusters: 137

Final distribution:
event_type
Speed Bump      205
Pothole         147
Road Bump         4
Hard Braking      2


In [21]:
EVENT_COLORS = {
    'Pothole':          'red',
    'Road Bump':        '#22AA22',
    'Speed Bump':       '#FFD700',
    'Hard Braking':     'orange',
    'Acceleration':     '#4488CC',
    'Rough Patch':      '#8B4513',
    'Possible Pothole': '#FF6600',
    'Minor Anomaly':    '#AAAAAA',
}
EVENT_OPACITY = {
    'Pothole':          0.90,
    'Road Bump':        0.55,
    'Speed Bump':       0.70,
    'Hard Braking':     0.35,
    'Acceleration':     0.30,
    'Rough Patch':      0.45,
    'Possible Pothole': 0.50,
    'Minor Anomaly':    0.15,
}
ROUTE_COLORS = {
    'Route_1': 'red',    'Route_2': 'blue',   'Route_3': 'green',
    'Route_4': 'orange', 'Route_5': 'purple',  'Route_6': 'darkred',
}

centre = [interesting['latitude'].mean(), interesting['longitude'].mean()]
m = folium.Map(location=centre, zoom_start=14, tiles='OpenStreetMap')

# Route polylines
for f in folders:
    loc = our_data[f]['location'].dropna(subset=['latitude','longitude'])
    coords = list(zip(loc['latitude'], loc['longitude']))
    folium.PolyLine(coords, color=ROUTE_COLORS[f], weight=4,
                    opacity=0.6, tooltip=f'{f}').add_to(m)

# Event markers
for _, row in interesting.iterrows():
    etype   = row['event_type']
    color   = EVENT_COLORS.get(etype, 'gray')
    opacity = EVENT_OPACITY.get(etype, 0.4)
    sev     = row['severity']
    conf    = row['confidence']

    if etype == 'Pothole':
        radius = max(8, min(20, sev * 2.2))
    elif etype in ('Speed Bump', 'Road Bump'):
        radius = max(5, min(12, sev * 1.3))
    else:
        radius = max(3, min(8, sev * 0.9))

    popup_html = (
        f'<b style="color:{color}">{etype}</b> | {row["route"]}<br>'
        f'Severity: {sev:.1f}/10 &nbsp; Conf: {conf:.0%}<br>'
        f'Pawar: {row["pawar_prob"]:.2f} &nbsp; '
        f'Speed: {row["speed"]:.1f} m/s ({row["speed"]*3.6:.0f} km/h)<br>'
        f'Gyro: {row["gyro_peak"]:.3f} rad/s &nbsp; '
        f'Kurtosis: {row["z_kurtosis"]:.1f}<br>'
        f'Z dip: {row["dip_magnitude"]:.2f} &nbsp; '
        f'Z bump: {row["bump_magnitude"]:.2f} m/s&sup2;<br>'
        f'X-accel: {row["accel_x_peak"]:.2f} m/s&sup2;<br>'
        f'<a href="https://www.google.com/maps?q='
        f'{row["latitude"]},{row["longitude"]}" '
        f'target="_blank">Open in Google Maps</a>'
    )

    folium.CircleMarker(
        [row['latitude'], row['longitude']],
        radius=radius,
        color=color, fill=True,
        fill_color=color, fill_opacity=opacity,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f'{etype} | sev={sev:.1f} | conf={conf:.0%}'
    ).add_to(m)

# Legend
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:9999;
            background:white; padding:12px; border:2px solid gray;
            border-radius:8px; font-size:13px; opacity:0.93;">
<b>Hybrid Pawar+V3 — Devyansh</b><br>
<br><b>Road Surface</b><br>
<span style="color:red">&#11044;</span> Pothole<br>
<span style="color:#22AA22">&#11044;</span> Road Bump<br>
<span style="color:#FFD700">&#11044;</span> Speed Bump<br>
<span style="color:#8B4513">&#11044;</span> Rough Patch<br>
<span style="color:#FF6600">&#11044;</span> Possible Pothole<br>
<br><b>Driver Behavior</b><br>
<span style="color:orange">&#11044;</span> Hard Braking<br>
<span style="color:#4488CC">&#11044;</span> Acceleration<br>
<br><b>Routes</b><br>
<span style="color:red">&#9473;</span> Route_1 &nbsp;
<span style="color:blue">&#9473;</span> Route_2 &nbsp;
<span style="color:green">&#9473;</span> Route_3<br>
<span style="color:orange">&#9473;</span> Route_4 &nbsp;
<span style="color:purple">&#9473;</span> Route_5 &nbsp;
<span style="color:darkred">&#9473;</span> Route_6
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

map_path = 'hybrid_pawar_v3_devyansh_map.html'
m.save(map_path)
print(f'Map saved: {map_path}')
print(f'Events on map: {len(interesting)}')
m

Map saved: hybrid_pawar_v3_devyansh_map.html
Events on map: 358


Map saved: hybrid_pawar_v3_devyansh_map.html
Events on map: 358


## 7. Export Results

In [22]:
export_cols = ['route','latitude','longitude','speed','event_type',
               'severity','confidence','pawar_prob','gyro_peak',
               'z_kurtosis','dip_magnitude','bump_magnitude',
               'accel_x_peak','total_z_dev','cluster']

csv_path = 'hybrid_events_devyansh.csv'
interesting[export_cols].to_csv(csv_path, index=False)
print(f'Exported {len(interesting)} events to {csv_path}')

print('\n' + '='*60)
print('FINAL SUMMARY')
print('='*60)
for f in folders:
    sub = interesting[interesting['route'] == f]
    if len(sub) == 0: continue
    print(f'\n{f}  ({len(sub)} events):')
    for et in sub['event_type'].value_counts().index:
        es = sub[sub['event_type'] == et]
        print(f'  {et:20s}: {len(es):3d}  '
              f'sev={es["severity"].mean():.1f}  '
              f'conf={es["confidence"].mean():.0%}  '
              f'gyro={es["gyro_peak"].mean():.3f}  '
              f'kurt={es["z_kurtosis"].mean():.1f}')

ph = interesting[interesting['event_type'] == 'Pothole']
bm = interesting[interesting['event_type'].isin(['Road Bump','Speed Bump'])]
print(f'\nTotal Potholes: {len(ph)}  |  Total Bumps: {len(bm)}  |  '
      f'Ratio: 1:{len(bm)/max(1,len(ph)):.1f}')
if len(ph) > 0:
    print(f'  Potholes — avg sev: {ph["severity"].mean():.1f}  '
          f'avg gyro: {ph["gyro_peak"].mean():.3f}  '
          f'avg kurt: {ph["z_kurtosis"].mean():.1f}')
if len(bm) > 0:
    print(f'  Bumps    — avg sev: {bm["severity"].mean():.1f}  '
          f'avg gyro: {bm["gyro_peak"].mean():.3f}  '
          f'avg kurt: {bm["z_kurtosis"].mean():.1f}')

Exported 358 events to hybrid_events_devyansh.csv

FINAL SUMMARY

Route_1  (38 events):
  Speed Bump          :  31  sev=5.8  conf=61%  gyro=0.262  kurt=0.1
  Pothole             :   7  sev=9.2  conf=95%  gyro=1.401  kurt=1.0

Route_2  (121 events):
  Speed Bump          :  69  sev=6.2  conf=59%  gyro=0.298  kurt=0.3
  Pothole             :  49  sev=8.2  conf=91%  gyro=1.590  kurt=3.3
  Road Bump           :   3  sev=1.9  conf=45%  gyro=0.364  kurt=0.2

Route_4  (83 events):
  Speed Bump          :  59  sev=4.7  conf=61%  gyro=0.286  kurt=0.3
  Pothole             :  21  sev=8.0  conf=89%  gyro=0.745  kurt=1.1
  Hard Braking        :   2  sev=2.0  conf=65%  gyro=0.203  kurt=0.1
  Road Bump           :   1  sev=2.0  conf=45%  gyro=0.319  kurt=0.3

Route_5  (43 events):
  Speed Bump          :  22  sev=5.2  conf=59%  gyro=0.335  kurt=0.3
  Pothole             :  21  sev=8.9  conf=95%  gyro=1.667  kurt=1.7

Route_6  (73 events):
  Pothole             :  49  sev=9.3  conf=95%  gyro=0.904  